# 05c — Filter Bad Photos

Replaces (or drops) legs whose photo was manually flagged as bad during visual QC.

**Why:** After NB07 reprojects the selected photos, `review_photos.py` is used to
manually screen them. Photos flagged as bad (e.g. obstructed view, construction,
poor quality) are written to `scripts/bad_photos.csv` with their `intersection_id`
and `leg_bearing`.

**Strategy per bad row (in order of preference):**
1. Find another leg of the same intersection in `leg_photo_selection_directional.csv`
   that is not back-direction AND not itself in `bad_photos.csv`. Prefer
   `front` > `left`/`right`; tiebreak on shortest `photo_dist_m`.
2. If no clean replacement leg exists: drop the row entirely — the intersection
   will not appear in the survey.

**Inputs:**
- `data/processed/sampled_legs_directional_noback.csv` — output of NB05b
- `data/processed/leg_photo_selection_directional.csv` — all legs (NB04 output)
- `scripts/bad_photos.csv` — manually flagged legs from `review_photos.py`

**Output:**
- `data/processed/sampled_legs_directional_clean.csv` — same schema, possibly
  fewer rows; added column `quality_patched`: `False` / `True` / `'dropped'`

**Note on bearing matching:** NB07 names files as `leg_{bearing:.0f}.jpeg` (rounded
to integer), so `bad_photos.csv` stores integer bearings. The selection CSVs store
float bearings. This notebook rounds floats before comparing.

**Depends on:** NB04, NB05, NB05b, and `review_photos.py` must be run first.

In [5]:
import os
import pandas as pd

PROJECT_DIR    = r"C:\Users\Thijs\OneDrive\Documents\Studie\EPA\Second_year\Afstuderen\Project\intersections"
PROCESSED_DIR  = os.path.join(PROJECT_DIR, "data", "processed")

# --- Image mode --- must match NB05 and NB05b
IMAGE_MODE     = "directional"  # change to "panorama" to process 360° panoramas

# Input: sampled legs after back-photo patch, all candidate legs, manual QC results
INPUT_CSV      = os.path.join(PROCESSED_DIR, f"sampled_legs_{IMAGE_MODE}_noback.csv")
SELECTION_CSV  = os.path.join(PROCESSED_DIR, f"leg_photo_selection_{IMAGE_MODE}.csv")
BAD_PHOTOS_CSV = os.path.join(PROJECT_DIR, "scripts", "bad_photos.csv")

# Output: cleaned sample, ready for NB06 and NB07
OUTPUT_CSV     = os.path.join(PROCESSED_DIR, f"sampled_legs_{IMAGE_MODE}_clean.csv")

# Columns that belong to the intersection, not the leg — keep from the original row
# when swapping in a replacement leg (same list as NB05b).
INTERSECTION_COLS = ["dim_type", "dim_risk", "dim_priority", "dim_speed", "is_centrum"]

# Direction preference when picking a replacement leg (lower = better)
DIRECTION_PREFERENCE = {"front": 0, "left": 1, "right": 2, "back": 3}

print(f"Input CSV   : {INPUT_CSV}")
print(f"Selection   : {SELECTION_CSV}")
print(f"Bad photos  : {BAD_PHOTOS_CSV}")
print(f"Output CSV  : {OUTPUT_CSV}")

Input CSV   : C:\Users\Thijs\OneDrive\Documents\Studie\EPA\Second_year\Afstuderen\Project\intersections\data\processed\sampled_legs_directional_noback.csv
Selection   : C:\Users\Thijs\OneDrive\Documents\Studie\EPA\Second_year\Afstuderen\Project\intersections\data\processed\leg_photo_selection_directional.csv
Bad photos  : C:\Users\Thijs\OneDrive\Documents\Studie\EPA\Second_year\Afstuderen\Project\intersections\scripts\bad_photos.csv
Output CSV  : C:\Users\Thijs\OneDrive\Documents\Studie\EPA\Second_year\Afstuderen\Project\intersections\data\processed\sampled_legs_directional_clean.csv


In [6]:
sampled   = pd.read_csv(INPUT_CSV)
selection = pd.read_csv(SELECTION_CSV)
bad       = pd.read_csv(BAD_PHOTOS_CSV)

# Build a set of (intersection_id, rounded_bearing) for O(1) lookup.
# bad_photos.csv uses integer bearings (parsed from filenames like leg_278.jpeg),
# so we round the float bearings in the selection CSV to match.
bad_set = set(zip(bad["intersection_id"].astype(int), bad["leg_bearing"].astype(int)))

print(f"Sampled legs    : {len(sampled)} rows")
print(f"Bad photos      : {len(bad)} rows flagged")
print(f"Full selection  : {len(selection)} rows across {selection['intersection_id'].nunique()} intersections")
print()
print("Direction breakdown before filter:")
print(sampled["selected_direction"].value_counts().to_string())

Sampled legs    : 368 rows
Bad photos      : 70 rows flagged
Full selection  : 12677 rows across 4615 intersections

Direction breakdown before filter:
selected_direction
front    349
left       8
back       6
right      5


In [7]:
def bearing_key(b) -> int:
    """Round a float bearing to int, matching NB07's leg_{bearing:.0f} filename convention."""
    return int(f"{b:.0f}")


def best_replacement(candidates: pd.DataFrame, bad_set: set):
    """Return the best clean replacement leg from a set of candidates, or None.

    A clean leg is one that is not back-direction and not in bad_photos.csv.
    Sorted by direction preference then by proximity to the intersection.
    """
    # Exclude back-direction legs and legs already flagged as bad
    clean = candidates[
        (candidates["selected_direction"] != "back") &
        (~candidates.apply(
            lambda r: (int(r["intersection_id"]), bearing_key(r["leg_bearing"])) in bad_set,
            axis=1
        ))
    ].copy()

    if clean.empty:
        return None

    clean["_dir_rank"] = clean["selected_direction"].map(DIRECTION_PREFERENCE).fillna(99)
    clean = clean.sort_values(["_dir_rank", "photo_dist_m"])
    return clean.iloc[0].drop(labels=["_dir_rank"])


# Index the full selection file by intersection_id for fast lookup
selection_by_intersection = {
    iid: grp for iid, grp in selection.groupby("intersection_id")
}

# object dtype so the column can hold False, True, and 'dropped' together
sampled["quality_patched"] = pd.array([False] * len(sampled), dtype=object)

rows_replaced = 0
drop_indices  = []   # rows with no clean replacement — will be removed

for idx, row in sampled.iterrows():
    row_key = (int(row["intersection_id"]), bearing_key(row["leg_bearing"]))

    # Skip rows whose photo is not flagged as bad
    if row_key not in bad_set:
        continue

    all_legs    = selection_by_intersection.get(row["intersection_id"], pd.DataFrame())
    replacement = best_replacement(all_legs, bad_set)

    if replacement is None:
        # No clean leg available — mark for removal
        sampled.at[idx, "quality_patched"] = "dropped"
        drop_indices.append(idx)
        print(f"  DROPPED   intersection {row['intersection_id']}  (no clean replacement)")
        continue

    # Overwrite leg-level columns with the replacement; preserve intersection-level columns
    for col in replacement.index:
        if col not in INTERSECTION_COLS:
            sampled.at[idx, col] = replacement[col]

    sampled.at[idx, "quality_patched"] = True
    rows_replaced += 1

# Remove rows that had no clean replacement
sampled_clean = sampled.drop(index=drop_indices).reset_index(drop=True)

print(f"\nResults:")
print(f"  Replaced (new leg used) : {rows_replaced}")
print(f"  Dropped (no clean leg)  : {len(drop_indices)}")
print(f"  Already clean           : {len(sampled) - rows_replaced - len(drop_indices)}")
print(f"\nOutput: {len(sampled_clean)} rows  (was {len(sampled)})")
print(f"\nDirection breakdown after filter:")
print(sampled_clean["selected_direction"].value_counts().to_string())

  DROPPED   intersection 176275201  (no clean replacement)
  DROPPED   intersection 177266031  (no clean replacement)
  DROPPED   intersection 177267087  (no clean replacement)
  DROPPED   intersection 177268025  (no clean replacement)
  DROPPED   intersection 177273178  (no clean replacement)
  DROPPED   intersection 180276079  (no clean replacement)
  DROPPED   intersection 181262039  (no clean replacement)
  DROPPED   intersection 181269032  (no clean replacement)
  DROPPED   intersection 181273011  (no clean replacement)
  DROPPED   intersection 182265016  (no clean replacement)
  DROPPED   intersection 182266047  (no clean replacement)
  DROPPED   intersection 183268173  (no clean replacement)
  DROPPED   intersection 183271208  (no clean replacement)
  DROPPED   intersection 183272048  (no clean replacement)
  DROPPED   intersection 184273085  (no clean replacement)
  DROPPED   intersection 185273022  (no clean replacement)
  DROPPED   intersection 185273146  (no clean replacemen

In [8]:
sampled_clean.to_csv(OUTPUT_CSV, index=False)
print(f"Saved {len(sampled_clean)} rows to {OUTPUT_CSV}")

Saved 329 rows to C:\Users\Thijs\OneDrive\Documents\Studie\EPA\Second_year\Afstuderen\Project\intersections\data\processed\sampled_legs_directional_clean.csv
